# 钢厂板坯设计问题

**类别：** 装箱

来源: [https://www.hexaly.com/templates/steel-mill-slab-design-problem](https://www.hexaly.com/templates/steel-mill-slab-design-problem)


## 问题

**在钢厂板坯设计问题中**，我们需要将钢铁订单的生产组织到板坯中。钢厂通过将铁水浇铸成板坯来生产钢材。钢厂可生产有限数量的板坯规格。每个订单具有两个属性：颜色（对应于钢厂中的某条工艺路径）和重量。板坯具有最大容量：分配给某块板坯的订单总重量不得超过该容量。此外，由于将板坯切割后再送往钢厂的不同工序代价高昂，因此每块板坯中所包含的不同颜色数量是有限制的（通常为两种）。目标是最小化钢材浪费，即生产出来但未被任何订单使用的钢材数量。更多详情请参见 [CSPLib](http://www.csplib.org/Problems/prob038/)。

	

### 学到的建模原则

- 使用 [set 决策变量](https://www.hexaly.com/docs/last/mathematicaloperators/collectionvariables.html) 对板坯进行建模
- 定义 [lambda 函数](https://www.hexaly.com/docs/last/mathematicaloperators/delegates.html) 来计算每块板坯所使用的钢材数量


## 数据

数据文件的格式如下：

- 第一行：板坯规格数以及所有可用的规格大小
- 第二行：颜色种类数
- 第三行：订单数量
- 接下来每一行描述一个订单：订单的规格大小和颜色

我们假设每块板坯最多只能包含两种不同颜色的订单。


## 模型

钢厂板坯设计问题的 Hexaly 模型使用 [set 决策变量](https://www.hexaly.com/docs/last/mathematicaloperators/collectionvariables.html) 来表示分配给每块板坯的订单集合。通过对集合变量使用 [**partition（划分）**](https://www.hexaly.com/docs/last/mathematicaloperators/collectionvariables.html#n-ary-operators) 运算符，我们约束每个订单恰好被分配到一块板坯中。

我们通过对集合使用可变参数数量的 **sum（求和）** 运算符以及一个返回订单大小的 [lambda 函数](https://www.hexaly.com/docs/last/mathematicaloperators/delegates.html) 来计算每块板坯所使用的钢材总数量。请注意，该求和的项数在搜索过程中会随着集合大小变化而变化。随后我们可以约束每块板坯使用的钢材总量不超过其最大规格大小。

利用 **distinct（去重）** 运算符和另一个 [lambda 函数](https://www.hexaly.com/docs/last/mathematicaloperators/delegates.html)，我们统计每块板坯中所包含的不同颜色数量。然后使用 **count（计数）** 运算符来确保每块板坯中所包含的最大颜色数得到满足。

最后，我们计算每块板坯的钢材浪费量，它等于能够容纳该板坯上所有订单的最小板坯规格，减去该板坯上各订单所使用的钢材总重量。


## Python 实现


In [ ]:
# Copyright (c) Hexaly. Permission is hereby granted to use, copy,
# and modify this code for applications developed with Hexaly.
import hexaly.optimizer
import sys

if len(sys.argv) < 2:
    print("Usage: python steel_mill_slab_design.py inputFile [outputFile] [timeLimit]")
    sys.exit(1)

def read_integers(filename):
    with open(filename) as f:
        return [int(elem) for elem in f.read().split()]


# Compute the vector waste_for_content
def pre_compute_waste_for_content(slab_sizes, sum_size_orders):
    # No waste when a slab is empty
    waste_for_content = [0] * sum_size_orders

    prev_size = 0
    for size in slab_sizes:
        if size < prev_size:
            print("Slab sizes should be sorted in ascending order")
            sys.exit(1)
        for content in range(prev_size + 1, size):
            waste_for_content[content] = size - content
        prev_size = size
    return waste_for_content


with hexaly.optimizer.HexalyOptimizer() as optimizer:
    #
    # Read instance data
    #
    nb_colors_max_slab = 2

    file_it = iter(read_integers(sys.argv[1]))
    nb_slab_sizes = next(file_it)
    slab_sizes = [next(file_it) for i in range(nb_slab_sizes)]
    max_size = slab_sizes[nb_slab_sizes - 1]

    nb_colors = next(file_it)
    nb_orders = next(file_it)
    nb_slabs = nb_orders

    sum_size_orders = 0

    # List of quantities and colors for each order
    quantities_data = []
    colors_data = []
    for o in range(nb_orders):
        quantities_data.append(next(file_it))
        colors_data.append(next(file_it))
        sum_size_orders += quantities_data[o]

    waste_for_content = pre_compute_waste_for_content(slab_sizes, sum_size_orders)

    #
    # Declare the optimization model
    #
    model = optimizer.model

    # Create array and function to retrieve the orders's colors and quantities
    colors = model.array(colors_data)
    color_lambda = model.lambda_function(lambda l: colors[l])
    quantities = model.array(quantities_data)
    quantity_lambda = model.lambda_function(lambda o: quantities[o])

    # Set decisions: slab[k] represents the orders in slab k
    slabs = [model.set(nb_orders) for s in range(nb_slabs)]

    # Each order must be in one slab and one slab only
    model.constraint(model.partition(slabs))

    slabContent = []
    
    for s in range(nb_slabs):

        # The number of colors per slab must not exceed a specified value
        model.constraint(model.count(model.distinct(slabs[s], color_lambda)) <= nb_colors_max_slab)

        # The content of each slab must not exceed the maximum size of the slab
        slabContent.append(model.sum(slabs[s], quantity_lambda))
        model.constraint(slabContent[s] <= max_size)

    waste_for_content_array = model.array(waste_for_content)

    # Wasted steel is computed according to the content of the slab
    wasted_steel = [waste_for_content_array[slabContent[s]] for s in range(nb_slabs)]

    # Minimize the total wasted steel
    total_wasted_steel = model.sum(wasted_steel)
    model.minimize(total_wasted_steel)

    model.close()

    # Parameterize the optimizer
    if len(sys.argv) >= 4:
        optimizer.param.time_limit = int(sys.argv[3])
    else:
        optimizer.param.time_limit = 60
    optimizer.solve()

    #
    # Write the solution in a file with the following format:
    #  - total wasted steel
    #  - number of slabs used
    #  - for each slab used, the number of orders in the slab and the list of orders
    #
    if len(sys.argv) >= 3:
        with open(sys.argv[2], 'w') as f:
            f.write("%d\n" % total_wasted_steel.value)
            actual_nb_slabs = 0
            for s in range(nb_slabs):
                if slabs[s].value.count() > 0:
                    actual_nb_slabs += 1
            f.write("%d\n" % actual_nb_slabs)

            for s in range(nb_slabs):
                nb_orders_in_slab = slabs[s].value.count()
                if nb_orders_in_slab == 0:
                    continue
                f.write("%d" % nb_orders_in_slab)
                for o in slabs[s].value:
                    f.write(" %d" % (o + 1))
                f.write("\n")
